# Dataset overview

Purpose: establish dataset coverage for Chapter 4 by reporting the observed hardware, model, workload, and experiment-category composition without assuming a fixed number of rows.

This notebook reads only `results/processed/final-analysis-dataset.csv` and writes derived figures and tables under `analysis/`. Missing measurements are excluded from the relevant calculation rather than replaced with zero.

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import load_dataset, save_figure, save_table, successful, grouped_bar, add_display_hardware

data = load_dataset(PROJECT_ROOT / "results" / "processed" / "final-analysis-dataset.csv")
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")

Loaded 630 rows and 37 columns


## Dataset shape

The shape is reported directly from the input so that rerunning the notebook after a legitimate dataset update does not require changing the analysis.

In [8]:
shape = pd.DataFrame({"measure": ["rows", "columns"], "value": [data.shape[0], data.shape[1]]})
display(shape)
save_table(shape, "01_dataset_shape.csv")

,measure,value
0,rows,630
1,columns,37


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/01_dataset_shape.csv')

## Hardware coverage

This table shows the number of executions and successful executions observed for each hardware platform.

In [9]:
hardware_coverage = (data.assign(hardware_label=data["hardware"].map({h: h for h in data["hardware"].unique()}))
    .groupby("hardware", as_index=False)
    .agg(executions=("experiment_id", "size"), successful=("status", lambda s: s.eq("success").sum()), models=("model", "nunique"), workloads=("workload", "nunique")))
hardware_coverage["success_rate"] = hardware_coverage["successful"] / hardware_coverage["executions"]
display(hardware_coverage)
save_table(hardware_coverage, "01_hardware_coverage.csv")

,hardware,executions,successful,models,workloads,success_rate
0,G3250,18,18,1,6,1.000000
1,gtx1650-4gb,72,72,4,6,1.000000
2,gtx1660-super-6gb,144,141,8,6,0.979167
3,i5-8265U,72,72,4,6,1.000000
4,i7-4790k,72,72,4,6,1.000000
5,rtx2060-12gb,162,162,9,6,1.000000
6,ryzen5600,90,90,5,6,1.000000


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/01_hardware_coverage.csv')

## Model coverage

Model coverage is reported across hardware and workload combinations to make gaps in the experimental matrix visible.

In [10]:
model_coverage = (data.groupby("model", as_index=False)
    .agg(executions=("experiment_id", "size"), hardware=("hardware", "nunique"), workloads=("workload", "nunique"), successful=("status", lambda s: s.eq("success").sum()), model_size_b=("model_size_b", "first"), architecture=("architecture", "first"), quantization=("quantization", "first")))
display(model_coverage)
save_table(model_coverage, "01_model_coverage.csv")

,model,executions,hardware,workloads,successful,model_size_b,architecture,quantization
0,Qwen3.5-0.8B,126,7,6,126,0.8,dense,Q8_0
1,Qwen3.5-2B,108,6,6,108,2.0,dense,Q4_K_M
2,Qwen3.5-4B,108,6,6,108,4.0,dense,Q4_K_M
3,Qwen3.5-9B,54,3,6,54,9.0,dense,Q4_K_M
4,Qwen3.6-27B,18,1,6,18,27.0,dense,Q4_K_M
5,Qwen3.6-35B-A3B,36,2,6,36,35.0,MoE,Q4_K_M
6,gemma-4-12B-it-QAT,36,2,6,36,12.0,dense,Q4_0
7,gemma-4-26B-A4B-it,36,2,6,33,26.0,MoE,Q4_K_M
8,gemma-4-E2B-it,108,6,6,108,2.0,MoE,Q4_K_M


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/01_model_coverage.csv')

## Workload coverage

The workload table documents the number of observations available for each task category.

In [11]:
workload_coverage = (data.groupby("workload", as_index=False)
    .agg(executions=("experiment_id", "size"), hardware=("hardware", "nunique"), models=("model", "nunique"), successful=("status", lambda s: s.eq("success").sum())))
display(workload_coverage)
save_table(workload_coverage, "01_workload_coverage.csv")

,workload,executions,hardware,models,successful
0,agentic,105,7,9,105
1,batch,105,7,9,105
2,chat,105,7,9,105
3,coding,105,7,9,105
4,summarization,105,7,9,102
5,world_knowledge,105,7,9,105


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/01_workload_coverage.csv')

## Experiment category distribution

Experiment categories are inherited from the reproducible dataset-generation step and are summarized here to distinguish CPU, full-offload, partial-offload, and MoE-offload observations.

In [12]:
category_coverage = data.groupby("experiment_category", as_index=False).agg(executions=("experiment_id", "size"), successful=("status", lambda s: s.eq("success").sum()))
display(category_coverage)
save_table(category_coverage, "01_experiment_category_coverage.csv")

,experiment_category,executions,successful
0,cpu_only,252,252
1,gpu_full_offload,270,270
2,gpu_moe_offload,72,69
3,gpu_partial_offload,36,36


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/01_experiment_category_coverage.csv')